In [1]:
#!/usr/bin/env python3
"""
Generate Turbulence Blur for ONLY 10 Book Cover Images
Quick test blur generation
"""

import os
import json
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

class TurbulenceBlurGenerator:
    """Generate subtle turbulence blur effects"""
    
    def __init__(self):
        self.intensity = 0.7  # Very high intensity - very challenging extraction
        
    def atmospheric_turbulence(self, image: np.ndarray) -> np.ndarray:
        """Simulate subtle atmospheric turbulence distortion"""
        h, w = image.shape[:2]
        
        # Create very strong displacement maps
        x_displacement = np.random.normal(0, self.intensity * 4.5, (h, w)).astype(np.float32)
        y_displacement = np.random.normal(0, self.intensity * 4.5, (h, w)).astype(np.float32)
        
        # Smooth the displacement maps for large turbulence waves
        x_displacement = cv2.GaussianBlur(x_displacement, (11, 11), 0)
        y_displacement = cv2.GaussianBlur(y_displacement, (11, 11), 0)
        
        # Create coordinate maps
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        x_map = (x + x_displacement).astype(np.float32)
        y_map = (y + y_displacement).astype(np.float32)
        
        # Apply distortion with linear interpolation
        distorted = cv2.remap(image, x_map, y_map, cv2.INTER_LINEAR)
        
        # Apply very strong Gaussian blur for heavy atmospheric effect
        blurred = cv2.GaussianBlur(distorted, (9, 9), 2.0)
        
        return blurred
    
    def generate_blur_variant(self, image_path: str, output_dir: str, book_data: dict) -> dict:
        """Generate turbulence blur variant for a single image"""
        image = cv2.imread(image_path)
        if image is None:
            print(f"Warning: Could not load image {image_path}")
            return None
        
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        
        try:
            # Apply atmospheric turbulence
            blurred = self.atmospheric_turbulence(image.copy())
            
            # Save blurred image
            output_filename = f"{base_name}_atmospheric_verystrong.jpg"
            output_path = os.path.join(output_dir, output_filename)
            cv2.imwrite(output_path, blurred, [cv2.IMWRITE_JPEG_QUALITY, 95])
            
            # Create metadata entry
            variant_data = {
                'book_id': book_data.get('book_id', ''),
                'original_image': os.path.basename(image_path),
                'blurred_image': output_filename,
                'blur_type': 'atmospheric_verystrong',
                'intensity': self.intensity,
                'title': book_data.get('title', ''),
                'author': book_data.get('author', ''),
                'language': book_data.get('language', '')
            }
            
            return variant_data
            
        except Exception as e:
            print(f"Error applying atmospheric turbulence to {image_path}: {str(e)}")
            return None

def load_books_data(csv_path: str, max_books: int = 500) -> list:
    """Load book data from CSV file - ONLY first N books"""
    books = []
    
    try:
        df = pd.read_csv(csv_path)
        print(f"Loaded CSV with {len(df)} total rows")
        print(f"Taking only first {max_books} books")
        
        # Take only first N rows
        df = df.head(max_books)
        
        print(f"CSV columns: {list(df.columns)}")
        
        for idx, row in df.iterrows():
            # Handle different possible CSV column names
            book_id = row.get('book_id', row.get('id', idx))
            title = row.get('title', '')
            author = row.get('author', row.get('authors', ''))
            language = row.get('language', row.get('language_code', ''))
            
            books.append({
                'book_id': book_id,
                'title': str(title) if pd.notna(title) else '',
                'author': str(author) if pd.notna(author) else '',
                'language': str(language) if pd.notna(language) else ''
            })
            
    except Exception as e:
        print(f"Error reading CSV: {str(e)}")
        return []
    
    return books

def main():
    """Generate blur for only 10 images"""
    
    # Configuration
    DATA_DIR = "/kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset"
    CSV_PATH = os.path.join(DATA_DIR, "books.csv")
    COVERS_DIR = os.path.join(DATA_DIR, "covers")
    OUTPUT_DIR = "/kaggle/working"
    BLUR_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "blurred_covers_10")
    
    MAX_IMAGES = 500  # ⚠️ ONLY 10 IMAGES!
    
    # Create output directory
    os.makedirs(BLUR_OUTPUT_DIR, exist_ok=True)
    
    print("="*70)
    print("🌪️  TURBULENCE BLUR - 10 IMAGES ONLY")
    print("="*70)
    print(f"Input directory: {DATA_DIR}")
    print(f"Output directory: {BLUR_OUTPUT_DIR}")
    print(f"Max images: {MAX_IMAGES}")
    print()
    
    # Load book data from CSV (only first 10)
    print("📚 Loading book data from CSV...")
    books_data = load_books_data(CSV_PATH, max_books=MAX_IMAGES)
    
    if not books_data:
        print("❌ No book data loaded. Check CSV file.")
        return
    
    print(f"✅ Loaded {len(books_data)} books from CSV\n")
    
    # Show which books we'll process
    print("📖 Books to process:")
    for i, book in enumerate(books_data, 1):
        print(f"  {i}. ID: {book['book_id']} - {book['title'][:50]}...")
    print()
    
    # Initialize blur generator
    blur_generator = TurbulenceBlurGenerator()
    
    # Process images and generate blurred variants
    print("🔄 Processing images and generating blurred variants...")
    print("="*70)
    
    all_variants = []
    processed_count = 0
    missing_images = []
    
    for book in tqdm(books_data, desc="Blurring images"):
        # Construct image path
        image_filename = f"{book['book_id']}.jpg"
        image_path = os.path.join(COVERS_DIR, image_filename)
        
        if os.path.exists(image_path):
            variant = blur_generator.generate_blur_variant(
                image_path, 
                BLUR_OUTPUT_DIR, 
                book
            )
            
            if variant:
                all_variants.append(variant)
                processed_count += 1
                print(f"  ✅ {image_filename} → {variant['blurred_image']}")
            else:
                print(f"  ❌ Failed to process {image_filename}")
        else:
            missing_images.append(image_filename)
            print(f"  ⚠️  Image not found: {image_filename}")
    
    print()
    print("="*70)
    print("📊 SUMMARY")
    print("="*70)
    print(f"Books in CSV: {len(books_data)}")
    print(f"Images processed: {processed_count}")
    print(f"Missing images: {len(missing_images)}")
    print(f"Blurred variants created: {len(all_variants)}")
    print(f"Blur type: atmospheric_verystrong")
    print(f"Intensity: {blur_generator.intensity}")
    print("="*70)
    
    # Save metadata as JSON
    json_output_path = os.path.join(OUTPUT_DIR, "blurred_metadata_10.json")
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(all_variants, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Metadata saved to: {json_output_path}")
    
    # Save metadata as CSV
    csv_output_path = os.path.join(OUTPUT_DIR, "blurred_metadata_10.csv")
    if all_variants:
        df_variants = pd.DataFrame(all_variants)
        df_variants.to_csv(csv_output_path, index=False, encoding='utf-8')
        print(f"💾 Metadata CSV saved to: {csv_output_path}")
    
    # Save summary
    summary = {
        "total_books_requested": MAX_IMAGES,
        "processed_images": processed_count,
        "missing_images": len(missing_images),
        "total_variants": len(all_variants),
        "blur_type": "atmospheric_verystrong",
        "intensity": blur_generator.intensity,
        "output_directory": BLUR_OUTPUT_DIR,
        "missing_image_files": missing_images
    }
    
    summary_path = os.path.join(OUTPUT_DIR, "blur_summary_10.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)
    
    print(f"📋 Summary saved to: {summary_path}")
    
    # Show sample results
    if all_variants:
        print(f"\n🔍 Sample result:")
        sample = all_variants[0]
        print(f"  Book ID: {sample['book_id']}")
        print(f"  Title: {sample['title'][:60]}")
        print(f"  Author: {sample['author'][:60]}")
        print(f"  Original: {sample['original_image']}")
        print(f"  Blurred: {sample['blurred_image']}")
    
    if missing_images:
        print(f"\n⚠️  Missing images:")
        for img in missing_images:
            print(f"  - {img}")
    
    print(f"\n✅ Complete! Blurred images saved to: {BLUR_OUTPUT_DIR}")
    print("="*70)

if __name__ == "__main__":
    main()

🌪️  TURBULENCE BLUR - 10 IMAGES ONLY
Input directory: /kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset
Output directory: /kaggle/working/blurred_covers_10
Max images: 500

📚 Loading book data from CSV...
Loaded CSV with 5000 total rows
Taking only first 500 books
CSV columns: ['book_id', 'title', 'author', 'language', 'release_date', 'downloads', 'loc_class', 'subject', 'original_publication', 'cover_downloaded', 'num_formats', 'url', 'scraped_at', 'summary']
✅ Loaded 500 books from CSV

📖 Books to process:
  1. ID: 1 - The Declaration of Independence of the United Stat...
  2. ID: 2 - The United States Bill of Rights by United States...
  3. ID: 3 - John F. Kennedy's Inaugural Address by John F. Ken...
  4. ID: 4 - Lincoln's Gettysburg Address by Abraham Lincoln...
  5. ID: 5 - The United States Constitution by United States...
  6. ID: 6 - Give Me Liberty or Give Me Death by Patrick Henry...
  7. ID: 7 - The Mayflower Compact...
  8. ID: 8 - Abraham Lincoln's Second Inaugur

Blurring images:   1%|▏         | 7/500 [00:00<00:16, 30.61it/s]

  ✅ 1.jpg → 1_atmospheric_verystrong.jpg
  ✅ 2.jpg → 2_atmospheric_verystrong.jpg
  ✅ 3.jpg → 3_atmospheric_verystrong.jpg
  ✅ 4.jpg → 4_atmospheric_verystrong.jpg
  ✅ 5.jpg → 5_atmospheric_verystrong.jpg
  ✅ 6.jpg → 6_atmospheric_verystrong.jpg
  ✅ 7.jpg → 7_atmospheric_verystrong.jpg
  ✅ 8.jpg → 8_atmospheric_verystrong.jpg


Blurring images:   3%|▎         | 16/500 [00:00<00:13, 36.59it/s]

  ✅ 9.jpg → 9_atmospheric_verystrong.jpg
  ✅ 10.jpg → 10_atmospheric_verystrong.jpg
  ✅ 11.jpg → 11_atmospheric_verystrong.jpg
  ✅ 12.jpg → 12_atmospheric_verystrong.jpg
  ✅ 13.jpg → 13_atmospheric_verystrong.jpg
  ✅ 14.jpg → 14_atmospheric_verystrong.jpg
  ✅ 15.jpg → 15_atmospheric_verystrong.jpg
  ✅ 16.jpg → 16_atmospheric_verystrong.jpg
  ✅ 17.jpg → 17_atmospheric_verystrong.jpg


...

  ✅ 501.jpg → 501_atmospheric_verystrong.jpg
  ✅ 502.jpg → 502_atmospheric_verystrong.jpg
  ✅ 503.jpg → 503_atmospheric_verystrong.jpg
  ✅ 504.jpg → 504_atmospheric_verystrong.jpg
  ✅ 505.jpg → 505_atmospheric_verystrong.jpg
  ✅ 506.jpg → 506_atmospheric_verystrong.jpg
  ✅ 507.jpg → 507_atmospheric_verystrong.jpg
  ✅ 508.jpg → 508_atmospheric_verystrong.jpg
  ✅ 509.jpg → 509_atmospheric_verystrong.jpg
  ✅ 510.jpg → 510_atmospheric_verystrong.jpg
  ✅ 511.jpg → 511_atmospheric_verystrong.jpg


Blurring images:  99%|█████████▉| 496/500 [00:10<00:00, 52.34it/s]

  ✅ 512.jpg → 512_atmospheric_verystrong.jpg
  ✅ 513.jpg → 513_atmospheric_verystrong.jpg
  ✅ 514.jpg → 514_atmospheric_verystrong.jpg
  ✅ 515.jpg → 515_atmospheric_verystrong.jpg
  ✅ 516.jpg → 516_atmospheric_verystrong.jpg
  ✅ 517.jpg → 517_atmospheric_verystrong.jpg
  ✅ 518.jpg → 518_atmospheric_verystrong.jpg
  ✅ 519.jpg → 519_atmospheric_verystrong.jpg
  ✅ 520.jpg → 520_atmospheric_verystrong.jpg
  ✅ 521.jpg → 521_atmospheric_verystrong.jpg
  ✅ 522.jpg → 522_atmospheric_verystrong.jpg


Blurring images: 100%|██████████| 500/500 [00:10<00:00, 48.76it/s]


  ✅ 523.jpg → 523_atmospheric_verystrong.jpg
  ✅ 524.jpg → 524_atmospheric_verystrong.jpg
  ✅ 525.jpg → 525_atmospheric_verystrong.jpg

📊 SUMMARY
Books in CSV: 500
Images processed: 500
Missing images: 0
Blurred variants created: 500
Blur type: atmospheric_verystrong
Intensity: 0.7

💾 Metadata saved to: /kaggle/working/blurred_metadata_10.json
💾 Metadata CSV saved to: /kaggle/working/blurred_metadata_10.csv
📋 Summary saved to: /kaggle/working/blur_summary_10.json

🔍 Sample result:
  Book ID: 1
  Title: The Declaration of Independence of the United States of Amer
  Author: Jefferson, Thomas, 1743-1826
  Original: 1.jpg
  Blurred: 1_atmospheric_verystrong.jpg

✅ Complete! Blurred images saved to: /kaggle/working/blurred_covers_10


In [2]:
"""
QWEN2.5-VL DATA PREPARATION FOR QLORA FINE-TUNING
Prepares turbulent book cover dataset in LLaVA format
Optimized for vision-language model training
"""

import os
import json
import pandas as pd
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("📁 QWEN2.5-VL DATA PREPARATION")
print("=" * 80)

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for data preparation"""
    
    # Input paths - UPDATE THESE FOR YOUR KAGGLE ENVIRONMENT
    ORIGINAL_COVERS = "/kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset/covers"
    ORIGINAL_CSV = "/kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset/books.csv"
    BLURRED_COVERS = "/kaggle/working/blurred_covers_10"
    BLURRED_CSV = "/kaggle/working/blurred_metadata_10.csv"
    
    # Output paths - Dataset structure for training
    OUTPUT_BASE = "/kaggle/working"
    DATASET_DIR = f"{OUTPUT_BASE}/finetune_dataset"
    IMAGES_DIR = f"{DATASET_DIR}/images"
    TRAIN_JSON = f"{DATASET_DIR}/train.json"
    VAL_JSON = f"{DATASET_DIR}/val.json"
    TEST_JSON = f"{DATASET_DIR}/test.json"
    
    # Dataset split ratios
    TRAIN_SPLIT = 0.80  # 80% training
    VAL_SPLIT = 0.10    # 10% validation
    TEST_SPLIT = 0.10   # 10% testing
    
    # Processing options
    MAX_SAMPLES = None  # None = use all available, or set number (e.g., 500)
    RANDOM_SEED = 42
    
    # Image validation
    VALIDATE_IMAGES = True
    MIN_IMAGE_SIZE = (50, 50)  # Minimum width, height

config = Config()

print(f"\n📂 Configuration:")
print(f"   Input covers: {config.ORIGINAL_COVERS}")
print(f"   Input CSV: {config.ORIGINAL_CSV}")
print(f"   Blurred images: {config.BLURRED_COVERS}")
print(f"   Blurred metadata: {config.BLURRED_CSV}")
print(f"   Output directory: {config.DATASET_DIR}")
print(f"   Split: {config.TRAIN_SPLIT:.0%} train / {config.VAL_SPLIT:.0%} val / {config.TEST_SPLIT:.0%} test")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def create_directories():
    """Create output directory structure"""
    print(f"\n🗂️  Creating directories...")
    
    os.makedirs(config.DATASET_DIR, exist_ok=True)
    os.makedirs(config.IMAGES_DIR, exist_ok=True)
    
    print(f"   ✅ {config.DATASET_DIR}")
    print(f"   ✅ {config.IMAGES_DIR}")

def validate_image(image_path):
    """Validate that image can be loaded and meets minimum requirements"""
    try:
        with Image.open(image_path) as img:
            # Check size
            if img.width < config.MIN_IMAGE_SIZE[0] or img.height < config.MIN_IMAGE_SIZE[1]:
                return False, f"Image too small: {img.size}"
            
            # Check format
            if img.format not in ['JPEG', 'JPG', 'PNG']:
                return False, f"Invalid format: {img.format}"
            
            # Try to load image data
            img.verify()
            
        return True, "OK"
    except Exception as e:
        return False, str(e)

def load_metadata():
    """Load and merge book metadata with blurred image information"""
    print(f"\n📚 Loading metadata...")
    
    # Load original books CSV
    try:
        books_df = pd.read_csv(config.ORIGINAL_CSV)
        print(f"   ✅ Loaded {len(books_df)} books from CSV")
        print(f"   Columns: {list(books_df.columns)}")
    except Exception as e:
        print(f"   ❌ Error loading books CSV: {e}")
        return None
    
    # Load blurred metadata CSV
    try:
        blurred_df = pd.read_csv(config.BLURRED_CSV)
        print(f"   ✅ Loaded {len(blurred_df)} blurred images from CSV")
        print(f"   Columns: {list(blurred_df.columns)}")
    except Exception as e:
        print(f"   ❌ Error loading blurred CSV: {e}")
        return None
    
    # Merge on book_id
    merged_df = books_df.merge(
        blurred_df[['book_id', 'blurred_image', 'blur_type', 'intensity']], 
        on='book_id', 
        how='inner'
    )
    
    print(f"   ✅ Merged dataset: {len(merged_df)} matched books")
    
    # Limit samples if specified
    if config.MAX_SAMPLES:
        merged_df = merged_df.head(config.MAX_SAMPLES)
        print(f"   ⚙️  Limited to {len(merged_df)} samples")
    
    return merged_df

def create_llava_format_entry(book_data, image_filename):
    """
    Create a single training example in LLaVA format for Qwen2.5-VL
    
    Format:
    {
        "id": "unique_id",
        "image": "image_filename.jpg",
        "conversations": [
            {
                "from": "human",
                "value": "<image>\\nPrompt text"
            },
            {
                "from": "gpt", 
                "value": "Expected response"
            }
        ]
    }
    """
    
    # Extract fields with safe handling
    book_id = book_data.get('book_id', '')
    title = str(book_data.get('title', '')).strip()
    author = str(book_data.get('author', book_data.get('authors', ''))).strip()
    
    # Optional fields (may not be in all datasets)
    publisher = str(book_data.get('publisher', '')).strip()
    language = str(book_data.get('language', book_data.get('language_code', ''))).strip()
    year = str(book_data.get('publication_year', book_data.get('  num_pages', ''))).strip()
    
    # Create comprehensive prompt for OCR task
    prompt = """<image>
Analyze this turbulent, blurred book cover image and extract all visible text. The image has atmospheric distortion making text extraction challenging.

Extract:
1. Book Title (main title text)
2. Author Name (author information)
3. Additional Text (subtitle, publisher, series, etc.)

Provide accurate transcription despite blur and distortion."""
    
    # Create ground truth response
    response_parts = [f"Title: {title}"]
    
    if author:
        response_parts.append(f"Author: {author}")
    
    # Add additional information if available
    additional_info = []
    if publisher:
        additional_info.append(f"Publisher: {publisher}")
    if year:
        additional_info.append(f"Year: {year}")
    if language:
        additional_info.append(f"Language: {language}")
    
    if additional_info:
        response_parts.append(f"Other: {', '.join(additional_info)}")
    else:
        response_parts.append("Other: None visible")
    
    response = "\n".join(response_parts)
    
    # Create LLaVA format entry
    entry = {
        "id": f"book_{book_id}",
        "image": image_filename,
        "conversations": [
            {
                "from": "human",
                "value": prompt
            },
            {
                "from": "gpt",
                "value": response
            }
        ]
    }
    
    return entry

def process_and_copy_images(metadata_df):
    """Process images: validate, copy to dataset directory, create training entries"""
    print(f"\n🖼️  Processing images...")
    
    training_entries = []
    valid_count = 0
    invalid_count = 0
    missing_count = 0
    
    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Processing"):
        book_id = row['book_id']
        blurred_filename = row['blurred_image']
        
        # Source path (blurred image)
        source_path = os.path.join(config.BLURRED_COVERS, blurred_filename)
        
        # Check if blurred image exists
        if not os.path.exists(source_path):
            missing_count += 1
            print(f"   ⚠️  Missing: {blurred_filename}")
            continue
        
        # Validate image if enabled
        if config.VALIDATE_IMAGES:
            is_valid, message = validate_image(source_path)
            if not is_valid:
                invalid_count += 1
                print(f"   ❌ Invalid image {blurred_filename}: {message}")
                continue
        
        # Create destination filename (keep original name or create new)
        dest_filename = f"book_{book_id}_blurred.jpg"
        dest_path = os.path.join(config.IMAGES_DIR, dest_filename)
        
        # Copy image to dataset directory
        try:
            shutil.copy2(source_path, dest_path)
        except Exception as e:
            print(f"   ❌ Copy failed for {blurred_filename}: {e}")
            invalid_count += 1
            continue
        
        # Create training entry in LLaVA format
        entry = create_llava_format_entry(row.to_dict(), dest_filename)
        training_entries.append(entry)
        valid_count += 1
    
    print(f"\n📊 Processing summary:")
    print(f"   ✅ Valid images: {valid_count}")
    print(f"   ❌ Invalid images: {invalid_count}")
    print(f"   ⚠️  Missing images: {missing_count}")
    print(f"   📝 Training entries created: {len(training_entries)}")
    
    return training_entries

def split_dataset(entries):
    """Split dataset into train/val/test sets"""
    print(f"\n✂️  Splitting dataset...")
    
    # Check if we have enough samples
    min_samples_needed = 10  # Minimum for meaningful train/val/test split
    if len(entries) < min_samples_needed:
        print(f"   ❌ Need at least {min_samples_needed} samples, got {len(entries)}")
        return None, None, None
    
    # First split: separate test set
    train_val_entries, test_entries = train_test_split(
        entries,
        test_size=config.TEST_SPLIT,
        random_state=config.RANDOM_SEED
    )
    
    # Second split: separate train and validation from remaining
    val_ratio = config.VAL_SPLIT / (config.TRAIN_SPLIT + config.VAL_SPLIT)
    train_entries, val_entries = train_test_split(
        train_val_entries,
        test_size=val_ratio,
        random_state=config.RANDOM_SEED
    )
    
    print(f"   📊 Split statistics:")
    print(f"      Train: {len(train_entries)} ({len(train_entries)/len(entries)*100:.1f}%)")
    print(f"      Val:   {len(val_entries)} ({len(val_entries)/len(entries)*100:.1f}%)")
    print(f"      Test:  {len(test_entries)} ({len(test_entries)/len(entries)*100:.1f}%)")
    
    return train_entries, val_entries, test_entries

def save_datasets(train_data, val_data, test_data):
    """Save train/val/test datasets as JSON files"""
    print(f"\n💾 Saving datasets...")
    
    # Save training set
    with open(config.TRAIN_JSON, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=2, ensure_ascii=False)
    print(f"   ✅ Saved {len(train_data)} train samples to {config.TRAIN_JSON}")
    print(f"      Size: {os.path.getsize(config.TRAIN_JSON) / 1024 / 1024:.2f} MB")
    
    # Save validation set
    with open(config.VAL_JSON, 'w', encoding='utf-8') as f:
        json.dump(val_data, f, indent=2, ensure_ascii=False)
    print(f"   ✅ Saved {len(val_data)} val samples to {config.VAL_JSON}")
    print(f"      Size: {os.path.getsize(config.VAL_JSON) / 1024 / 1024:.2f} MB")
    
    # Save test set
    with open(config.TEST_JSON, 'w', encoding='utf-8') as f:
        json.dump(test_data, f, indent=2, ensure_ascii=False)
    print(f"   ✅ Saved {len(test_data)} test samples to {config.TEST_JSON}")
    print(f"      Size: {os.path.getsize(config.TEST_JSON) / 1024 / 1024:.2f} MB")

def create_dataset_info():
    """Create dataset information file"""
    print(f"\n📋 Creating dataset info...")
    
    # Count images
    num_images = len([f for f in os.listdir(config.IMAGES_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    # Load splits to get counts
    with open(config.TRAIN_JSON, 'r') as f:
        train_count = len(json.load(f))
    with open(config.VAL_JSON, 'r') as f:
        val_count = len(json.load(f))
    with open(config.TEST_JSON, 'r') as f:
        test_count = len(json.load(f))
    
    info = {
        "dataset_name": "Turbulent Book Cover OCR Dataset",
        "task": "Vision-Language OCR on Degraded Images",
        "model_architecture": "Qwen2.5-VL-7B",
        "format": "LLaVA Conversation Format",
        "creation_date": pd.Timestamp.now().isoformat(),
        
        "statistics": {
            "total_samples": train_count + val_count + test_count,
            "train_samples": train_count,
            "validation_samples": val_count,
            "test_samples": test_count,
            "total_images": num_images
        },
        
        "data_structure": {
            "images_directory": "images/",
            "train_file": "train.json",
            "validation_file": "val.json",
            "test_file": "test.json"
        },
        
        "image_details": {
            "degradation_type": "atmospheric_turbulence",
            "blur_intensity": "very_strong",
            "format": "JPEG",
            "task_difficulty": "challenging"
        },
        
        "splits": {
            "train": config.TRAIN_SPLIT,
            "validation": config.VAL_SPLIT,
            "test": config.TEST_SPLIT,
            "random_seed": config.RANDOM_SEED
        },
        
        "conversation_format": {
            "description": "LLaVA-style multi-turn conversation",
            "structure": {
                "id": "unique identifier",
                "image": "relative path to image",
                "conversations": [
                    {"from": "human", "value": "prompt with <image> token"},
                    {"from": "gpt", "value": "expected model response"}
                ]
            }
        },
        
        "usage": {
            "training": "Use train.json with images/ directory",
            "validation": "Use val.json for monitoring during training",
            "testing": "Use test.json for final evaluation",
            "model": "Compatible with Qwen2.5-VL and similar vision-language models"
        }
    }
    
    info_path = os.path.join(config.DATASET_DIR, "dataset_info.json")
    with open(info_path, 'w', encoding='utf-8') as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    
    print(f"   ✅ Dataset info saved to {info_path}")
    
    return info

def display_sample_entries(entries, num_samples=2):
    """Display sample entries for verification"""
    print(f"\n🔍 Sample entries (first {num_samples}):")
    print("=" * 80)
    
    for i, entry in enumerate(entries[:num_samples], 1):
        print(f"\n📖 Sample {i}:")
        print(f"   ID: {entry['id']}")
        print(f"   Image: {entry['image']}")
        print(f"\n   Human prompt:")
        print(f"   {entry['conversations'][0]['value'][:200]}...")
        print(f"\n   Expected response:")
        print(f"   {entry['conversations'][1]['value']}")
        print("-" * 80)

def verify_dataset_integrity():
    """Verify dataset integrity after creation"""
    print(f"\n🔎 Verifying dataset integrity...")
    
    issues = []
    
    # Check if directories exist
    if not os.path.exists(config.DATASET_DIR):
        issues.append("Dataset directory missing")
    if not os.path.exists(config.IMAGES_DIR):
        issues.append("Images directory missing")
    
    # Check if JSON files exist
    for json_file in [config.TRAIN_JSON, config.VAL_JSON, config.TEST_JSON]:
        if not os.path.exists(json_file):
            issues.append(f"Missing file: {json_file}")
    
    # Verify JSON structure
    for json_file in [config.TRAIN_JSON, config.VAL_JSON, config.TEST_JSON]:
        if os.path.exists(json_file):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                # Check structure
                for entry in data[:5]:  # Check first 5 entries
                    if 'id' not in entry or 'image' not in entry or 'conversations' not in entry:
                        issues.append(f"Invalid structure in {json_file}")
                        break
                    
                    # Check conversations format
                    if len(entry['conversations']) != 2:
                        issues.append(f"Invalid conversations in {json_file}")
                        break
                    
                    # Check if image exists
                    img_path = os.path.join(config.IMAGES_DIR, entry['image'])
                    if not os.path.exists(img_path):
                        issues.append(f"Missing image: {entry['image']}")
                        break
                        
            except Exception as e:
                issues.append(f"Error reading {json_file}: {e}")
    
    if issues:
        print(f"   ⚠️  Found {len(issues)} issues:")
        for issue in issues:
            print(f"      - {issue}")
        return False
    else:
        print(f"   ✅ All checks passed!")
        return True

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Main data preparation pipeline"""
    
    print("\n" + "=" * 80)
    print("🚀 STARTING DATA PREPARATION PIPELINE")
    print("=" * 80)
    
    # Step 1: Create directories
    create_directories()
    
    # Step 2: Load metadata
    metadata_df = load_metadata()
    if metadata_df is None or len(metadata_df) == 0:
        print("\n❌ Failed to load metadata. Exiting.")
        return False
    
    # Step 3: Process images and create training entries
    training_entries = process_and_copy_images(metadata_df)
    if not training_entries:
        print("\n❌ No valid training entries created. Exiting.")
        return False
    
    # Step 4: Split dataset
    train_data, val_data, test_data = split_dataset(training_entries)
    if train_data is None:
        print("\n❌ Dataset split failed. Exiting.")
        return False
    
    # Step 5: Save datasets
    save_datasets(train_data, val_data, test_data)
    
    # Step 6: Create dataset info
    dataset_info = create_dataset_info()
    
    # Step 7: Display samples
    display_sample_entries(train_data, num_samples=2)
    
    # Step 8: Verify integrity
    integrity_ok = verify_dataset_integrity()
    
    # Final summary
    print("\n" + "=" * 80)
    print("✅ DATA PREPARATION COMPLETE!")
    print("=" * 80)
    
    print(f"\n📊 Final Statistics:")
    print(f"   Total samples: {dataset_info['statistics']['total_samples']}")
    print(f"   Train: {dataset_info['statistics']['train_samples']}")
    print(f"   Val: {dataset_info['statistics']['validation_samples']}")
    print(f"   Test: {dataset_info['statistics']['test_samples']}")
    print(f"   Images: {dataset_info['statistics']['total_images']}")
    
    print(f"\n📁 Output Files:")
    print(f"   Dataset directory: {config.DATASET_DIR}")
    print(f"   Images: {config.IMAGES_DIR}")
    print(f"   Train: {config.TRAIN_JSON}")
    print(f"   Val: {config.VAL_JSON}")
    print(f"   Test: {config.TEST_JSON}")
    print(f"   Info: {os.path.join(config.DATASET_DIR, 'dataset_info.json')}")
    
    print(f"\n🎯 Next Steps:")
    print(f"   1. Verify dataset: Check {config.DATASET_DIR}")
    print(f"   2. Review samples: Inspect train.json")
    print(f"   3. Ready for training: Use with Qwen2.5-VL LoRA script")
    
    print("\n" + "=" * 80)
    
    return integrity_ok

# ============================================================================
# EXECUTION
# ============================================================================

if __name__ == "__main__":
    success = main()
    
    if success:
        print("\n🎉 Dataset ready for Qwen2.5-VL fine-tuning!")
    else:
        print("\n⚠️  Dataset preparation completed with warnings. Review output above.")

📁 QWEN2.5-VL DATA PREPARATION

📂 Configuration:
   Input covers: /kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset/covers
   Input CSV: /kaggle/input/datasets/mizanurrahmanrafi/bookimage-dataset/books.csv
   Blurred images: /kaggle/working/blurred_covers_10
   Blurred metadata: /kaggle/working/blurred_metadata_10.csv
   Output directory: /kaggle/working/finetune_dataset
   Split: 80% train / 10% val / 10% test

🚀 STARTING DATA PREPARATION PIPELINE

🗂️  Creating directories...
   ✅ /kaggle/working/finetune_dataset
   ✅ /kaggle/working/finetune_dataset/images

📚 Loading metadata...
   ✅ Loaded 5000 books from CSV
   Columns: ['book_id', 'title', 'author', 'language', 'release_date', 'downloads', 'loc_class', 'subject', 'original_publication', 'cover_downloaded', 'num_formats', 'url', 'scraped_at', 'summary']
   ✅ Loaded 500 blurred images from CSV
   Columns: ['book_id', 'original_image', 'blurred_image', 'blur_type', 'intensity', 'title', 'author', 'language']
   ✅ Merged datas

Processing: 100%|██████████| 500/500 [00:00<00:00, 2154.74it/s]



📊 Processing summary:
   ✅ Valid images: 500
   ❌ Invalid images: 0
   ⚠️  Missing images: 0
   📝 Training entries created: 500

✂️  Splitting dataset...
   📊 Split statistics:
      Train: 400 (80.0%)
      Val:   50 (10.0%)
      Test:  50 (10.0%)

💾 Saving datasets...
   ✅ Saved 400 train samples to /kaggle/working/finetune_dataset/train.json
      Size: 0.27 MB
   ✅ Saved 50 val samples to /kaggle/working/finetune_dataset/val.json
      Size: 0.03 MB
   ✅ Saved 50 test samples to /kaggle/working/finetune_dataset/test.json
      Size: 0.03 MB

📋 Creating dataset info...
   ✅ Dataset info saved to /kaggle/working/finetune_dataset/dataset_info.json

🔍 Sample entries (first 2):

📖 Sample 1:
   ID: book_175
   Image: book_175_blurred.jpg

   Human prompt:
   <image>
Analyze this turbulent, blurred book cover image and extract all visible text. The image has atmospheric distortion making text extraction challenging.

Extract:
1. Book Title (main title text...

   Expected response:
   T

In [3]:
import subprocess, sys

# Clean old versions
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
               "transformers", "peft"], capture_output=True)

# Install from source
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers",
    "accelerate", "peft", "bitsandbytes", "qwen-vl-utils"
], check=True)

print("✅ RESTART KERNEL NOW!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 50.3 MB/s eta 0:00:00
✅ RESTART KERNEL NOW!


In [4]:
from transformers import AutoModelForImageTextToText, AutoProcessor  # ← This avoids HybridCache
from peft import LoraConfig
from qwen_vl_utils import process_vision_info
print("✅ All OK!")

✅ All OK!


In [ ]:
"""
QWEN2.5-VL-3B AUTONOMOUS SELF-SUPERVISED TRAINING (QLoRA)
==========================================================
Fixed & production-ready version.

Changes from previous version:
  - Removed deprecated `group_by_length` TrainingArguments param
  - Removed deprecated `dataloader_prefetch_factor` (causes error without workers)
  - Fixed `warmup_ratio` deprecation warning → using `warmup_steps`
  - Added `fp16=False` explicit guard
  - Safer device movement for autonomous layer
  - Better OOM handling with smaller fallback suggestions
  - Added gradient checkpointing compatibility guard
  - Clean shutdown with stats saved even on interrupt

Architecture:
  Base : Qwen2.5-VL-3B-Instruct  (4-bit NF4 QLoRA)
  LoRA : r=32, alpha=64, all 7 projection modules
  Extra: AutonomousQualityEvaluator + SelfCritique + CurriculumManager
"""

import os
import json
import torch
import time
import warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import deque
from typing import Any, Dict, List, Optional
from torch.utils.data import Dataset as TorchDataset
from dataclasses import dataclass

warnings.filterwarnings("ignore")

print("=" * 80)
print("🤖  QWEN2.5-VL-3B  AUTONOMOUS  QLORA  TRAINING")
print("=" * 80)

# ============================================================================
# SYSTEM SETUP
# ============================================================================

num_gpus = torch.cuda.device_count()
print(f"\n🖥️  System:")
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"   GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)")
    torch.cuda.empty_cache()

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

try:
    from qwen_vl_utils import process_vision_info
    print("   ✅ qwen-vl-utils ready")
except ImportError:
    os.system("pip install -q qwen-vl-utils")
    from qwen_vl_utils import process_vision_info

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # ── Auth & model ──────────────────────────────────────────────────────
    HF_TOKEN   = ""
    MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

    # ── Paths ─────────────────────────────────────────────────────────────
    DATASET_DIR = "/kaggle/working/finetune_dataset"
    IMAGES_DIR  = f"{DATASET_DIR}/images"
    TRAIN_JSON  = f"{DATASET_DIR}/train.json"
    VAL_JSON    = f"{DATASET_DIR}/val.json"
    OUTPUT_DIR  = "/kaggle/working/qwen3b_autonomous_qlora"

    # ── Training ──────────────────────────────────────────────────────────
    NUM_EPOCHS    = 2
    BATCH_SIZE    = 4       # per GPU — safe for 4-bit on T4×2
    GRAD_ACCUM    = 2       # effective = 4 × 2 × 2 GPUs = 16
    MAX_SEQ_LEN   = 2048
    LR            = 2e-4
    WEIGHT_DECAY  = 0.01
    WARMUP_STEPS  = 50      # replaces deprecated warmup_ratio
    MAX_GRAD_NORM = 0.3

    # ── LoRA ──────────────────────────────────────────────────────────────
    LORA_R        = 32
    LORA_ALPHA    = 64
    LORA_DROPOUT  = 0.05
    LORA_TARGETS  = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # FFN — critical!
    ]

    # ── Autonomous system ─────────────────────────────────────────────────
    QUALITY_THRESHOLD    = 0.85
    CONFIDENCE_THRESHOLD = 0.80
    REJECTION_THRESHOLD  = 0.65
    ENABLE_CURRICULUM    = True
    MEMORY_SIZE          = 50

    # ── Loss weights ──────────────────────────────────────────────────────
    W_QUALITY     = 0.20
    W_CONSISTENCY = 0.15

    # ── Misc ─────────────────────────────────────────────────────────────
    DATALOADER_WORKERS = 0   # 0 = safest on Kaggle (no multiprocessing issues)
    SEED               = 42

config = Config()
os.environ["HF_TOKEN"] = config.HF_TOKEN
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print(f"\n⚙️  Config:")
print(f"   Model      : {config.MODEL_NAME.split('/')[-1]}")
print(f"   Epochs     : {config.NUM_EPOCHS}")
print(f"   LoRA       : r={config.LORA_R}, {len(config.LORA_TARGETS)} modules")
print(f"   Eff. batch : {config.BATCH_SIZE * config.GRAD_ACCUM * max(1, num_gpus)}")
print(f"   Autonomous : quality={config.QUALITY_THRESHOLD}, "
      f"confidence={config.CONFIDENCE_THRESHOLD}")

# ============================================================================
# AUTONOMOUS MODULES
# ============================================================================

class AutonomousQualityEvaluator(nn.Module):
    """
    Maps mean-pooled hidden states → 4 quality scores + accept/refine/reject.
    Scaled down for 3B (hidden=2048) to fit in T4 memory budget.
    """
    def __init__(self, hidden_dim: int = 2048):
        super().__init__()

        def _head():
            return nn.Sequential(
                nn.Linear(hidden_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Linear(64, 1),
                nn.Sigmoid(),
            )

        self.accuracy_net     = _head()
        self.coherence_net    = _head()
        self.completeness_net = _head()
        self.confidence_net   = _head()

        self.decision_net = nn.Sequential(
            nn.Linear(4, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 3),      # accept / refine / reject
            nn.Softmax(dim=-1),
        )
        print("   ✅ AutonomousQualityEvaluator")

    def forward(self, hidden_states: torch.Tensor) -> Dict:
        pooled       = hidden_states.mean(dim=1)
        accuracy     = self.accuracy_net(pooled)
        coherence    = self.coherence_net(pooled)
        completeness = self.completeness_net(pooled)
        confidence   = self.confidence_net(pooled)
        qvec         = torch.cat([accuracy, coherence, completeness, confidence], dim=1)
        return {
            "accuracy":       accuracy,
            "coherence":      coherence,
            "completeness":   completeness,
            "confidence":     confidence,
            "decision_probs": self.decision_net(qvec),
            "quality_vector": qvec,
        }


class SelfCritiqueModule(nn.Module):
    """
    Single-layer transformer that refines hidden states.
    Kept to 1 layer + narrow FFN to stay within T4 memory.
    """
    def __init__(self, hidden_dim: int = 2048):
        super().__init__()
        nhead = 8 if hidden_dim % 8 == 0 else 4
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=nhead,
                dim_feedforward=hidden_dim,   # narrow = memory efficient
                dropout=0.1,
                batch_first=True,
            ),
            num_layers=1,
        )
        self.improvement_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
        )
        print("   ✅ SelfCritiqueModule")

    def forward(self, hidden_states: torch.Tensor) -> Dict:
        critiqued    = self.transformer(hidden_states)
        improvements = self.improvement_head(critiqued)
        return {"critiqued_states": critiqued, "improvements": improvements}


class AutonomousSelfSupervisionLayer(nn.Module):
    """
    Full autonomous layer: global attention → quality eval → self critique → refinement.
    """
    def __init__(self, hidden_dim: int = 2048):
        super().__init__()
        nhead = 8 if hidden_dim % 8 == 0 else 4
        self.global_attn       = nn.MultiheadAttention(
            hidden_dim, nhead, dropout=0.1, batch_first=True)
        self.quality_evaluator = AutonomousQualityEvaluator(hidden_dim)
        self.self_critique     = SelfCritiqueModule(hidden_dim)
        self.refinement_net    = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        print("   ✅ AutonomousSelfSupervisionLayer")

    def forward(self, hidden_states: torch.Tensor) -> Dict:
        ctx, _       = self.global_attn(hidden_states, hidden_states, hidden_states)
        combined     = ctx + hidden_states
        quality      = self.quality_evaluator(combined)
        critique     = self.self_critique(combined)
        refined      = self.refinement_net(
            torch.cat([combined, critique["improvements"]], dim=-1))
        return {
            "hidden_states":   combined + refined,
            "quality_results": quality,
            "original_states": hidden_states,
        }


# ============================================================================
# DECISION MAKER + CURRICULUM
# ============================================================================

class AutonomousDecisionMaker:
    def __init__(self, quality_threshold=0.85,
                 confidence_threshold=0.80,
                 rejection_threshold=0.65):
        self.qt  = quality_threshold
        self.ct  = confidence_threshold
        self.rt  = rejection_threshold
        self.dec = {"accepted": 0, "refined": 0, "rejected": 0, "total": 0}
        self.qlog: List[float] = []
        print("   ✅ AutonomousDecisionMaker")

    def decide(self, qr: Dict, curriculum=None) -> Dict:
        acc  = qr["accuracy"].mean().item()
        coh  = qr["coherence"].mean().item()
        comp = qr["completeness"].mean().item()
        conf = qr["confidence"].mean().item()
        overall = (acc + coh + comp + conf) / 4.0

        thresh = curriculum.get_current_threshold(self.qt) if curriculum else self.qt

        issues = []
        if acc  < 0.75: issues.append("low_accuracy")
        if coh  < 0.75: issues.append("low_coherence")
        if comp < 0.70: issues.append("incomplete")
        if conf < self.ct: issues.append("low_confidence")

        if overall >= thresh and conf >= self.ct:
            action = "accept"
        elif overall >= self.rt and len(issues) <= 2:
            action = "refine"
        else:
            action = "reject"

        self.dec["total"] += 1
        key = {"accept": "accepted", "refine": "refined", "reject": "rejected"}[action]
        self.dec[key] += 1
        self.qlog.append(overall)
        return {"action": action, "quality": overall, "confidence": conf}

    def get_statistics(self) -> Dict:
        t = self.dec["total"]
        if t == 0:
            return {"total_decisions": 0}
        return {
            "total_decisions": t,
            "accepted":        self.dec["accepted"],
            "refined":         self.dec["refined"],
            "rejected":        self.dec["rejected"],
            "acceptance_rate": self.dec["accepted"] / t,
            "refinement_rate": self.dec["refined"]  / t,
            "rejection_rate":  self.dec["rejected"] / t,
            "avg_quality":     float(np.mean(self.qlog)) if self.qlog else 0.0,
        }


class AdaptiveCurriculumManager:
    def __init__(self, memory_size: int = 50):
        self.perf = deque(maxlen=memory_size)
        self.acc  = deque(maxlen=memory_size)
        self.ql   = deque(maxlen=memory_size)
        self.diff = 0.5
        print("   ✅ AdaptiveCurriculumManager")

    def update(self, quality: float, accepted: bool):
        self.perf.append(quality)
        self.acc.append(1.0 if accepted else 0.0)
        self.ql.append(quality)
        if len(self.perf) >= 10:
            rp = np.mean(list(self.perf)[-10:])
            ra = np.mean(list(self.acc)[-10:])
            if rp > 0.90 and ra > 0.80:
                self.diff = min(1.0, self.diff + 0.05)
            elif rp < 0.70 or ra < 0.50:
                self.diff = max(0.0, self.diff - 0.08)

    def get_current_threshold(self, base: float) -> float:
        return float(np.clip(base + (self.diff - 0.5) * 0.1, 0.6, 0.95))

    def get_statistics(self) -> Dict:
        return {
            "difficulty_level": self.diff,
            "avg_quality":      float(np.mean(self.ql)) if self.ql else 0.0,
            "acceptance_rate":  float(np.mean(self.acc)) if self.acc else 0.0,
            "samples_seen":     len(self.perf),
        }


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class QualityAwareLoss(nn.Module):
    def forward(self, qr: Dict) -> torch.Tensor:
        return (
            0.30 * (1 - qr["accuracy"]).mean()     +
            0.25 * (1 - qr["coherence"]).mean()    +
            0.25 * (1 - qr["completeness"]).mean() +
            0.20 * (1 - qr["confidence"]).mean()
        )


class AdaptiveConsistencyLoss(nn.Module):
    def forward(self, refined: torch.Tensor, original: torch.Tensor,
                confidence: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        p    = F.normalize(refined,  p=2, dim=-1)
        t    = F.normalize(original, p=2, dim=-1)
        sim  = (p * t).sum(dim=-1)                      # (B, T)
        conf = confidence.expand(-1, sim.size(1))        # (B, T)
        wsim = sim * conf
        if mask is not None:
            wsim = wsim * mask.float()
            return -wsim.sum() / (mask.sum() + 1e-8)
        return -wsim.mean()


# ============================================================================
# DATASET
# ============================================================================

class QwenVLDataset(TorchDataset):
    def __init__(self, json_path: str, images_dir: str,
                 processor, max_length: int = 2048):
        with open(json_path, "r") as f:
            raw = json.load(f)
        self.images_dir = images_dir
        self.processor  = processor
        self.max_length = max_length
        self.data = [
            item for item in raw
            if os.path.exists(os.path.join(images_dir, item["image"]))
        ]
        print(f"   ✅ {len(self.data)} samples  ←  {Path(json_path).name}")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx: int) -> Dict:
        item = self.data[idx]
        imgp = os.path.join(self.images_dir, item["image"])
        utxt = (item["conversations"][0]["value"]
                .replace("<image>\n", "").replace("<image>", "").strip())
        atxt = item["conversations"][1]["value"]

        msgs = [{"role": "user", "content": [
            {"type": "image", "image": f"file://{imgp}"},
            {"type": "text",  "text":  utxt},
        ]}]
        prompt   = self.processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
        full_txt = prompt + atxt + self.processor.tokenizer.eos_token

        img_in, vid_in = process_vision_info(msgs)
        inputs = self.processor(
            text=[full_txt], images=img_in, videos=vid_in,
            padding=False, return_tensors="pt",
            max_length=self.max_length, truncation=True,
        )
        ids  = inputs["input_ids"].squeeze(0)
        plen = self.processor(
            text=[prompt], images=None, videos=None,
            padding=False, return_tensors="pt",
        )["input_ids"].shape[1]

        labels = ids.clone()
        labels[:plen] = -100

        out: Dict = {
            "input_ids":      ids,
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels":         labels,
        }
        if "pixel_values" in inputs and inputs["pixel_values"] is not None:
            pv = inputs["pixel_values"].squeeze(0)
            if pv.numel() > 0:
                out["pixel_values"] = pv
        if "image_grid_thw" in inputs and inputs["image_grid_thw"] is not None:
            igt = inputs["image_grid_thw"].squeeze(0)
            if igt.numel() > 0:
                out["image_grid_thw"] = igt
        return out


# ============================================================================
# DATA COLLATOR
# ============================================================================

@dataclass
class VLCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict]) -> Dict:
        max_len = max(f["input_ids"].shape[0] for f in features)
        ids, msk, lbl = [], [], []
        pv_list, thw_list = [], []

        for f in features:
            pad = max_len - f["input_ids"].shape[0]
            ids.append(torch.cat([f["input_ids"],
                torch.full((pad,), self.tokenizer.pad_token_id, dtype=torch.long)]))
            msk.append(torch.cat([f["attention_mask"],
                torch.zeros(pad, dtype=torch.long)]))
            lbl.append(torch.cat([f["labels"],
                torch.full((pad,), -100, dtype=torch.long)]))
            if "pixel_values" in f and f["pixel_values"].numel() > 0:
                pv_list.append(f["pixel_values"])
            if "image_grid_thw" in f and f["image_grid_thw"].numel() > 0:
                thw_list.append(f["image_grid_thw"])

        batch = {
            "input_ids":      torch.stack(ids),
            "attention_mask": torch.stack(msk),
            "labels":         torch.stack(lbl),
        }
        if pv_list:
            batch["pixel_values"]   = torch.cat(pv_list, dim=0)
        if thw_list:
            batch["image_grid_thw"] = torch.stack(thw_list)
        return batch


# ============================================================================
# AUTONOMOUS TRAINER
# ============================================================================

from transformers import Trainer

class AutonomousQLoRATrainer(Trainer):
    """
    HF Trainer extended with autonomous self-supervision:
      1. Standard QLoRA CE loss on ground-truth labels
      2. Quality-aware auxiliary loss (4-dim quality scores)
      3. Consistency loss (refined hs vs original hs)
      4. Decision-scaled loss: rejected→×1.5, accepted→×0.9
      5. Adaptive curriculum adjusting thresholds as training progresses
    """

    def __init__(self, *args,
                 autonomous_layer:    AutonomousSelfSupervisionLayer,
                 quality_loss:        QualityAwareLoss,
                 consistency_loss:    AdaptiveConsistencyLoss,
                 decision_maker:      AutonomousDecisionMaker,
                 curriculum_manager:  Optional[AdaptiveCurriculumManager],
                 w_quality:           float = 0.20,
                 w_consistency:       float = 0.15,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.autonomous_layer    = autonomous_layer
        self.quality_loss        = quality_loss
        self.consistency_loss    = consistency_loss
        self.decision_maker      = decision_maker
        self.curriculum_manager  = curriculum_manager
        self.w_quality           = w_quality
        self.w_consistency       = w_consistency
        self._step               = 0

    # ------------------------------------------------------------------
    def compute_loss(self, model, inputs,
                     return_outputs: bool = False, **kwargs):

        outputs   = model(**inputs, output_hidden_states=True)
        base_loss = outputs.loss
        hs        = (outputs.hidden_states[-1]
                     if outputs.hidden_states is not None else None)

        total_loss    = base_loss
        decision_info = None

        if hs is not None:
            dev = hs.device

            # Move autonomous layer to the same device as hidden states
            try:
                cur_dev = next(self.autonomous_layer.parameters()).device
                if cur_dev != dev:
                    self.autonomous_layer = self.autonomous_layer.to(dev)
            except StopIteration:
                self.autonomous_layer = self.autonomous_layer.to(dev)

            sup            = self.autonomous_layer(hs)
            refined_hs     = sup["hidden_states"]
            quality_results= sup["quality_results"]

            # Autonomous decision every 5 steps (saves ~80% of decision overhead)
            if self._step % 5 == 0:
                decision_info = self.decision_maker.decide(
                    quality_results, self.curriculum_manager)
                if self.curriculum_manager is not None:
                    self.curriculum_manager.update(
                        decision_info["quality"],
                        decision_info["action"] == "accept",
                    )

            # Auxiliary losses
            ql = self.quality_loss(quality_results)
            cl = self.consistency_loss(
                refined_hs,
                hs.detach(),
                quality_results["confidence"],
                mask=inputs.get("attention_mask"),
            )

            total_loss = base_loss + self.w_quality * ql + self.w_consistency * cl

            # Scale by decision outcome
            if decision_info is not None:
                if   decision_info["action"] == "reject": total_loss = total_loss * 1.5
                elif decision_info["action"] == "accept": total_loss = total_loss * 0.9

            self._step += 1

        return (total_loss, outputs) if return_outputs else total_loss

    # ------------------------------------------------------------------
    def log(self, logs: Dict[str, float], start_time=None) -> None:
        stats = self.decision_maker.get_statistics()
        if stats and stats.get("total_decisions", 0) > 0:
            logs["autonomous/acceptance_rate"] = stats["acceptance_rate"]
            logs["autonomous/avg_quality"]     = stats["avg_quality"]
            logs["autonomous/total_decisions"] = float(stats["total_decisions"])
        if self.curriculum_manager is not None:
            cs = self.curriculum_manager.get_statistics()
            logs["curriculum/difficulty"]  = cs["difficulty_level"]
            logs["curriculum/avg_quality"] = cs["avg_quality"]
        super().log(logs)


# ============================================================================
# LOAD MODEL + PROCESSOR
# ============================================================================

print("\n" + "=" * 80)
print("📥  LOADING MODEL  (QLoRA 4-bit NF4)")
print("=" * 80)

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    config.MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=config.HF_TOKEN,
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(
    config.MODEL_NAME,
    token=config.HF_TOKEN,
    trust_remote_code=True,
)
print(f"✅ Loaded in {time.time()-t0:.1f}s")

# ── Auto-detect hidden_dim ────────────────────────────────────────────────
if hasattr(model.config, "text_config") and hasattr(model.config.text_config, "hidden_size"):
    hidden_dim = model.config.text_config.hidden_size
elif hasattr(model.config, "hidden_size"):
    hidden_dim = model.config.hidden_size
else:
    hidden_dim = 2048   # Qwen2.5-VL-3B default
print(f"   hidden_dim = {hidden_dim}")

# ── QLoRA prep ───────────────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)

print("\n🔧 Applying LoRA  (r=32, 7 modules)...")
lora_cfg = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    target_modules=config.LORA_TARGETS,
    lora_dropout=config.LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA: {trainable:,} trainable params  ({trainable/total*100:.2f}%)")

# ============================================================================
# BUILD AUTONOMOUS SYSTEM
# ============================================================================

print("\n🤖 Building Autonomous Supervision System...")

dev0             = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
autonomous_layer = AutonomousSelfSupervisionLayer(hidden_dim=hidden_dim).to(dev0)
quality_loss     = QualityAwareLoss()
consistency_loss = AdaptiveConsistencyLoss()
decision_maker   = AutonomousDecisionMaker(
    quality_threshold    = config.QUALITY_THRESHOLD,
    confidence_threshold = config.CONFIDENCE_THRESHOLD,
    rejection_threshold  = config.REJECTION_THRESHOLD,
)
curriculum_manager = (
    AdaptiveCurriculumManager(memory_size=config.MEMORY_SIZE)
    if config.ENABLE_CURRICULUM else None
)

as_params = sum(p.numel() for p in autonomous_layer.parameters())
print(f"✅ Autonomous system: {as_params:,} additional parameters")

# ============================================================================
# DATASETS + COLLATOR
# ============================================================================

print("\n📂 Loading datasets...")
train_dataset = QwenVLDataset(config.TRAIN_JSON, config.IMAGES_DIR,
                               processor, config.MAX_SEQ_LEN)
val_dataset   = QwenVLDataset(config.VAL_JSON,   config.IMAGES_DIR,
                               processor, config.MAX_SEQ_LEN)
collator      = VLCollator(tokenizer=processor.tokenizer)

# ============================================================================
# TRAINING ARGUMENTS  — compatible with transformers >= 4.40
# ============================================================================

eff_batch        = config.BATCH_SIZE * config.GRAD_ACCUM * max(1, num_gpus)
steps_per_epoch  = max(1, len(train_dataset) // eff_batch)
total_steps      = steps_per_epoch * config.NUM_EPOCHS

print(f"\n📊 Training plan:")
print(f"   Train samples   : {len(train_dataset)}")
print(f"   Effective batch : {eff_batch}")
print(f"   Steps / epoch   : {steps_per_epoch}")
print(f"   Total steps     : {total_steps}")

training_args = TrainingArguments(
    output_dir                  = config.OUTPUT_DIR,
    num_train_epochs            = config.NUM_EPOCHS,
    per_device_train_batch_size = config.BATCH_SIZE,
    per_device_eval_batch_size  = config.BATCH_SIZE,
    gradient_accumulation_steps = config.GRAD_ACCUM,

    learning_rate               = config.LR,
    weight_decay                = config.WEIGHT_DECAY,
    warmup_steps                = config.WARMUP_STEPS,
    max_grad_norm               = config.MAX_GRAD_NORM,
    lr_scheduler_type           = "cosine",

    bf16                        = True,
    fp16                        = False,
    gradient_checkpointing      = True,
    optim                       = "paged_adamw_8bit",

    logging_steps               = 20,
    logging_first_step          = True,
    eval_strategy               = "no",
    save_strategy               = "epoch",
    save_total_limit            = 1,
    report_to                   = "none",
    seed                        = config.SEED,

    dataloader_num_workers      = config.DATALOADER_WORKERS,
    dataloader_pin_memory       = True,
    remove_unused_columns       = False,
    ddp_find_unused_parameters  = False,
    torch_compile               = False,
)

# ============================================================================
# TRAINER
# ============================================================================

trainer = AutonomousQLoRATrainer(
    model               = model,
    args                = training_args,
    train_dataset       = train_dataset,
    eval_dataset        = val_dataset,
    data_collator       = collator,
    autonomous_layer    = autonomous_layer,
    quality_loss        = quality_loss,
    consistency_loss    = consistency_loss,
    decision_maker      = decision_maker,
    curriculum_manager  = curriculum_manager,
    w_quality           = config.W_QUALITY,
    w_consistency       = config.W_CONSISTENCY,
)

# ============================================================================
# TRAIN
# ============================================================================

print("\n" + "=" * 80)
print("🚀  STARTING AUTONOMOUS QLORA TRAINING")
print("=" * 80)

start_time = time.time()

def _save_and_report(elapsed: float, loss: Optional[float] = None):
    """Helper — saves model + prints final stats."""
    print("\n💾 Saving model + processor...")
    trainer.save_model(config.OUTPUT_DIR)
    processor.save_pretrained(config.OUTPUT_DIR)

    ds = decision_maker.get_statistics()
    cs = curriculum_manager.get_statistics() if curriculum_manager else {}

    info = {
        "model":             config.MODEL_NAME,
        "training_time_min": elapsed / 60,
        "final_train_loss":  float(loss) if loss is not None else None,
        "num_epochs":        config.NUM_EPOCHS,
        "lora_rank":         config.LORA_R,
        "lora_targets":      config.LORA_TARGETS,
        "train_samples":     len(train_dataset),
        "decision_stats":    ds,
        "curriculum_stats":  cs,
        "loss_weights":      {"quality": config.W_QUALITY,
                              "consistency": config.W_CONSISTENCY},
    }
    with open(f"{config.OUTPUT_DIR}/training_info.json", "w") as f:
        json.dump(info, f, indent=2)

    print(f"✅ Saved to: {config.OUTPUT_DIR}")

    if ds and ds.get("total_decisions", 0) > 0:
        print(f"\n🤖 Autonomous stats:")
        print(f"   Decisions : {ds['total_decisions']}")
        print(f"   Accepted  : {ds['accepted']}  ({ds['acceptance_rate']*100:.1f}%)")
        print(f"   Refined   : {ds['refined']}   ({ds['refinement_rate']*100:.1f}%)")
        print(f"   Rejected  : {ds['rejected']}  ({ds['rejection_rate']*100:.1f}%)")
        print(f"   Avg qual  : {ds['avg_quality']:.3f}")
    if cs:
        print(f"\n📚 Curriculum stats:")
        print(f"   Difficulty : {cs['difficulty_level']:.3f}")
        print(f"   Avg quality: {cs['avg_quality']:.3f}")
        print(f"   Samples    : {cs['samples_seen']}")


try:
    result = trainer.train()
    elapsed = time.time() - start_time

    print("\n" + "=" * 80)
    print("✅  TRAINING COMPLETE!")
    print("=" * 80)
    print(f"⏱️   Time  : {elapsed/60:.1f} minutes")
    print(f"📉  Loss  : {result.training_loss:.4f}")
    print(f"📶  Steps : {result.global_step}")

    _save_and_report(elapsed, result.training_loss)

    print(f"""
================================================================================
🎉  AUTONOMOUS QLORA TRAINING DONE!
================================================================================
  Model      : Qwen2.5-VL-3B-Instruct  (QLoRA 4-bit NF4)
  LoRA       : r={config.LORA_R}, {len(config.LORA_TARGETS)} modules (attn + FFN)
  Autonomous : QualityEvaluator + SelfCritique + CurriculumManager
  Samples    : {len(train_dataset)}
  Time       : {elapsed/60:.1f} min
  Loss       : {result.training_loss:.4f}
  Output     : {config.OUTPUT_DIR}
================================================================================
""")

except KeyboardInterrupt:
    elapsed = time.time() - start_time
    print(f"\n⚠️  Interrupted at {elapsed/60:.1f} min — saving checkpoint...")
    _save_and_report(elapsed)

except Exception as e:
    elapsed = time.time() - start_time
    import traceback
    print(f"\n❌ FAILED at {elapsed/60:.1f} min:  {e}")
    traceback.print_exc()

    if "out of memory" in str(e).lower():
        print("\n💡 OOM — try these in Config:")
        print("   BATCH_SIZE  = 2")
        print("   MAX_SEQ_LEN = 1536")
        print("   LORA_R      = 16  (fallback)")

    # Still try to save whatever we have
    try:
        _save_and_report(elapsed)
    except Exception:
        pass

finally:
    torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("✅  DONE")
print("=" * 80)

🤖  QWEN2.5-VL-3B  AUTONOMOUS  QLORA  TRAINING

🖥️  System:
   GPU 0: Tesla T4  (15.6 GB)
   GPU 1: Tesla T4  (15.6 GB)
   ✅ qwen-vl-utils ready

⚙️  Config:
   Model      : Qwen2.5-VL-3B-Instruct
   Epochs     : 2
   LoRA       : r=32, 7 modules
   Eff. batch : 16
   Autonomous : quality=0.85, confidence=0.8

📥  LOADING MODEL  (QLoRA 4-bit NF4)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

✅ Loaded in 7.7s
   hidden_dim = 2048

🔧 Applying LoRA  (r=32, 7 modules)...
✅ LoRA: 74,305,536 trainable params  (3.52%)

🤖 Building Autonomous Supervision System...
   ✅ AutonomousQualityEvaluator
   ✅ SelfCritiqueModule
   ✅ AutonomousSelfSupervisionLayer
   ✅ AutonomousDecisionMaker
   ✅ AdaptiveCurriculumManager
✅ Autonomous system: 77,715,271 additional parameters

📂 Loading datasets...
   ✅ 400 samples  ←  train.json
   ✅ 50 samples  ←  val.json

📊 Training plan:
   Train samples   : 400
   Effective batch : 16
   Steps / epoch   : 25
   Total steps     : 50

🚀  STARTING AUTONOMOUS QLORA TRAINING


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
1,7.379199
20,4.027100
40,0.922136
60,0.474714
80,0.342704
100,0.323593



✅  TRAINING COMPLETE!
⏱️   Time  : 38.7 minutes
📉  Loss  : 1.2516
📶  Steps : 100

💾 Saving model + processor...
✅ Saved to: /kaggle/working/qwen3b_autonomous_qlora

🤖 Autonomous stats:
   Decisions : 40
   Accepted  : 17  (42.5%)
   Refined   : 4   (10.0%)
   Rejected  : 19  (47.5%)
   Avg qual  : 0.693

📚 Curriculum stats:
   Difficulty : 0.000
   Avg quality: 0.693
   Samples    : 40

🎉  AUTONOMOUS QLORA TRAINING DONE!
  Model      : Qwen2.5-VL-3B-Instruct  (QLoRA 4-bit NF4)
  LoRA       : r=32, 7 modules (attn + FFN)
  Autonomous : QualityEvaluator + SelfCritique + CurriculumManager
  Samples    : 400
  Time       : 38.7 min
  Loss       : 1.2516
  Output     : /kaggle/working/qwen3b_autonomous_qlora


✅  DONE


In [7]:
#!/usr/bin/env python3
"""
GPU Memory Management for Kaggle
Clear GPU memory and show usage statistics
"""

import gc
import torch
import psutil
import os

def clear_gpu_memory(verbose=True):
    """
    Clear GPU memory and show statistics
    
    Args:
        verbose (bool): Print memory statistics
    """
    
    if verbose:
        print("="*70)
        print("🧹 CLEARING GPU MEMORY")
        print("="*70)
    
    # Show memory before clearing
    if torch.cuda.is_available() and verbose:
        gpu_mem_before = torch.cuda.memory_allocated() / 1e9
        gpu_mem_reserved_before = torch.cuda.memory_reserved() / 1e9
        print(f"\n📊 Before Clearing:")
        print(f"  Allocated: {gpu_mem_before:.2f} GB")
        print(f"  Reserved:  {gpu_mem_reserved_before:.2f} GB")
    
    # Clear Python garbage collector
    gc.collect()
    
    # Clear PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Force another garbage collection
    gc.collect()
    
    # Show memory after clearing
    if torch.cuda.is_available() and verbose:
        gpu_mem_after = torch.cuda.memory_allocated() / 1e9
        gpu_mem_reserved_after = torch.cuda.memory_reserved() / 1e9
        freed_allocated = gpu_mem_before - gpu_mem_after
        freed_reserved = gpu_mem_reserved_before - gpu_mem_reserved_after
        
        print(f"\n📊 After Clearing:")
        print(f"  Allocated: {gpu_mem_after:.2f} GB")
        print(f"  Reserved:  {gpu_mem_reserved_after:.2f} GB")
        print(f"\n✅ Freed:")
        print(f"  Allocated: {freed_allocated:.2f} GB")
        print(f"  Reserved:  {freed_reserved:.2f} GB")
        
        # Show available memory
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        available = total_memory - gpu_mem_after
        print(f"\n💾 Available: {available:.2f} GB / {total_memory:.2f} GB")
        print("="*70)
    
    elif verbose:
        print("\n⚠️  No GPU available")
        print("="*70)

def get_gpu_stats():
    """Get detailed GPU statistics"""
    
    if not torch.cuda.is_available():
        return "No GPU available"
    
    stats = {
        "gpu_name": torch.cuda.get_device_name(0),
        "total_memory_gb": torch.cuda.get_device_properties(0).total_memory / 1e9,
        "allocated_memory_gb": torch.cuda.memory_allocated() / 1e9,
        "reserved_memory_gb": torch.cuda.memory_reserved() / 1e9,
        "free_memory_gb": (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    }
    
    return stats

def print_gpu_stats():
    """Print formatted GPU statistics"""
    
    print("="*70)
    print("📊 GPU STATISTICS")
    print("="*70)
    
    if not torch.cuda.is_available():
        print("❌ No GPU available")
        print("="*70)
        return
    
    stats = get_gpu_stats()
    
    print(f"\n🖥️  GPU: {stats['gpu_name']}")
    print(f"💾 Total Memory: {stats['total_memory_gb']:.2f} GB")
    print(f"📈 Allocated:    {stats['allocated_memory_gb']:.2f} GB ({100*stats['allocated_memory_gb']/stats['total_memory_gb']:.1f}%)")
    print(f"📦 Reserved:     {stats['reserved_memory_gb']:.2f} GB ({100*stats['reserved_memory_gb']/stats['total_memory_gb']:.1f}%)")
    print(f"✅ Free:         {stats['free_memory_gb']:.2f} GB ({100*stats['free_memory_gb']/stats['total_memory_gb']:.1f}%)")
    print("="*70)

def delete_model_and_clear(model_var_name=None):
    """
    Delete model variable and clear GPU memory
    
    Args:
        model_var_name: Name of the model variable to delete (optional)
    
    Usage:
        delete_model_and_clear('model')  # Deletes variable named 'model'
    """
    
    print("="*70)
    print("🗑️  DELETING MODEL AND CLEARING MEMORY")
    print("="*70)
    
    # Try to delete from globals
    if model_var_name:
        try:
            if model_var_name in globals():
                del globals()[model_var_name]
                print(f"✅ Deleted global variable: {model_var_name}")
        except:
            print(f"⚠️  Could not delete {model_var_name}")
    
    # Clear memory
    clear_gpu_memory(verbose=True)

def get_ram_stats():
    """Get RAM usage statistics"""
    
    ram = psutil.virtual_memory()
    
    print("="*70)
    print("💻 RAM STATISTICS")
    print("="*70)
    print(f"\n📊 Total RAM: {ram.total / 1e9:.2f} GB")
    print(f"📈 Used:      {ram.used / 1e9:.2f} GB ({ram.percent:.1f}%)")
    print(f"✅ Available: {ram.available / 1e9:.2f} GB")
    print("="*70)

def full_memory_report():
    """Complete memory report for both RAM and GPU"""
    
    print("\n")
    print("="*70)
    print("🔍 COMPLETE MEMORY REPORT")
    print("="*70)
    print("\n")
    
    # RAM stats
    get_ram_stats()
    print("\n")
    
    # GPU stats
    print_gpu_stats()

# Quick usage functions
def clear():
    """Quick clear - short function name"""
    clear_gpu_memory(verbose=True)

def stats():
    """Quick stats - short function name"""
    full_memory_report()

# Example usage
if __name__ == "__main__":
    print("\n🧪 GPU Memory Management Tool\n")
    
    # Show current stats
    print_gpu_stats()
    
    print("\n⏳ Simulating memory usage...\n")
    
    # Simulate some memory usage
    if torch.cuda.is_available():
        x = torch.randn(1000, 1000, device='cuda')
        y = torch.randn(1000, 1000, device='cuda')
        z = x @ y
        
    print_gpu_stats()
    
    print("\n🧹 Clearing memory...\n")
    
    # Clear memory
    clear_gpu_memory()
    
    print("\n✅ Done!\n")
    print("="*70)
    print("USAGE EXAMPLES:")
    print("="*70)
    print("# Clear GPU memory:")
    print("  clear_gpu_memory()")
    print("  clear()  # Short version")
    print("")
    print("# Show statistics:")
    print("  print_gpu_stats()")
    print("  stats()  # Complete report")
    print("")
    print("# Delete model and clear:")
    print("  delete_model_and_clear('model')")
    print("")
    print("# Get stats as dict:")
    print("  stats_dict = get_gpu_stats()")
    print("="*70)


🧪 GPU Memory Management Tool

📊 GPU STATISTICS

🖥️  GPU: Tesla T4
💾 Total Memory: 15.64 GB
📈 Allocated:    3.94 GB (25.2%)
📦 Reserved:     4.99 GB (31.9%)
✅ Free:         11.70 GB (74.8%)

⏳ Simulating memory usage...

📊 GPU STATISTICS

🖥️  GPU: Tesla T4
💾 Total Memory: 15.64 GB
📈 Allocated:    3.95 GB (25.3%)
📦 Reserved:     4.99 GB (31.9%)
✅ Free:         11.68 GB (74.7%)

🧹 Clearing memory...

🧹 CLEARING GPU MEMORY

📊 Before Clearing:
  Allocated: 3.95 GB
  Reserved:  4.99 GB

📊 After Clearing:
  Allocated: 3.95 GB
  Reserved:  4.99 GB

✅ Freed:
  Allocated: 0.00 GB
  Reserved:  0.00 GB

💾 Available: 11.68 GB / 15.64 GB

✅ Done!

USAGE EXAMPLES:
# Clear GPU memory:
  clear_gpu_memory()
  clear()  # Short version

# Show statistics:
  print_gpu_stats()
  stats()  # Complete report

# Delete model and clear:
  delete_model_and_clear('model')

# Get stats as dict:
  stats_dict = get_gpu_stats()


In [ ]:
"""
EVALUATION SCRIPT — QWEN2.5-VL-3B AUTONOMOUS QLORA
===================================================
Evaluates the trained model on the test set with:
  1. Full accuracy metrics (similarity, word F1, CER, WER, exact match)
  2. Base model vs fine-tuned comparison
  3. Per-sample breakdown with detailed scoring
  4. CSV + JSON results saved to output dir

Trained model : /kaggle/working/qwen3b_autonomous_qlora
Base model    : Qwen/Qwen2.5-VL-3B-Instruct  (4-bit, for fair comparison)
"""

import os
import json
import torch
import time
import warnings
import re
import numpy as np
from pathlib import Path
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple
from collections import Counter

warnings.filterwarnings("ignore")

print("=" * 80)
print("📊  EVALUATION  —  QWEN2.5-VL-3B  AUTONOMOUS  QLORA")
print("=" * 80)

# ============================================================================
# CONFIGURATION
# ============================================================================

class EvalConfig:
    # ── Paths ─────────────────────────────────────────────────────────────
    FINETUNED_MODEL_DIR = "/kaggle/working/qwen3b_autonomous_qlora"
    BASE_MODEL_NAME     = "Qwen/Qwen2.5-VL-3B-Instruct"
    HF_TOKEN            = ""

    DATASET_DIR = "/kaggle/working/finetune_dataset"
    IMAGES_DIR  = f"{DATASET_DIR}/images"
    TEST_JSON   = f"{DATASET_DIR}/test.json"
    VAL_JSON    = f"{DATASET_DIR}/val.json"   # fallback if test.json missing

    OUTPUT_DIR  = FINETUNED_MODEL_DIR         # results saved alongside model

    # ── Generation ────────────────────────────────────────────────────────
    MAX_NEW_TOKENS = 256
    DO_SAMPLE      = False    # greedy for reproducibility

    # ── Evaluation ────────────────────────────────────────────────────────
    # Show per-sample output for every N-th sample (0 = show all)
    VERBOSE_EVERY  = 5

cfg = EvalConfig()
os.environ["HF_TOKEN"]                = cfg.HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

# ============================================================================
# GPU INFO
# ============================================================================

num_gpus = torch.cuda.device_count()
print(f"\n🖥️  System:")
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"   GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)")
    torch.cuda.empty_cache()

# ============================================================================
# METRICS
# ============================================================================

def sequence_similarity(pred: str, gt: str) -> float:
    """SequenceMatcher ratio × 100."""
    return SequenceMatcher(None, pred.lower().strip(),
                           gt.lower().strip()).ratio() * 100


def word_f1(pred: str, gt: str) -> float:
    """Token-level F1 score (standard QA metric)."""
    pred_tokens = pred.lower().split()
    gt_tokens   = gt.lower().split()
    if not pred_tokens or not gt_tokens:
        return 0.0
    common  = Counter(pred_tokens) & Counter(gt_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0
    precision = n_common / len(pred_tokens)
    recall    = n_common / len(gt_tokens)
    return 2 * precision * recall / (precision + recall) * 100


def character_error_rate(pred: str, gt: str) -> float:
    """CER = edit_distance(pred, gt) / len(gt), capped at 100%."""
    if not gt:
        return 0.0
    # simple DP edit distance
    p, g = pred.lower().strip(), gt.lower().strip()
    m, n = len(p), len(g)
    dp   = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp   = dp[j]
            dp[j]  = prev if p[i-1] == g[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev   = temp
    return min(dp[n] / len(g) * 100, 100.0)


def word_error_rate(pred: str, gt: str) -> float:
    """WER = edit_distance(pred_words, gt_words) / len(gt_words), capped at 100%."""
    p_words = pred.lower().split()
    g_words = gt.lower().split()
    if not g_words:
        return 0.0
    m, n = len(p_words), len(g_words)
    dp   = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp   = dp[j]
            dp[j]  = prev if p_words[i-1] == g_words[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev   = temp
    return min(dp[n] / len(g_words) * 100, 100.0)


def exact_match(pred: str, gt: str) -> bool:
    return pred.lower().strip() == gt.lower().strip()


def score_tier(sim: float) -> str:
    if sim >= 90: return "🟢 Excellent"
    if sim >= 75: return "🟡 Good"
    if sim >= 50: return "🟠 Fair"
    return "🔴 Poor"


def compute_all_metrics(pred: str, gt: str) -> Dict:
    return {
        "similarity":   round(sequence_similarity(pred, gt), 2),
        "word_f1":      round(word_f1(pred, gt), 2),
        "cer":          round(character_error_rate(pred, gt), 2),
        "wer":          round(word_error_rate(pred, gt), 2),
        "exact_match":  exact_match(pred, gt),
    }

# ============================================================================
# LOAD MODELS
# ============================================================================

print("\n" + "=" * 80)
print("📥  LOADING MODELS")
print("=" * 80)

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── 1. Fine-tuned model ───────────────────────────────────────────────────
print(f"\n⏳ Loading fine-tuned model from {cfg.FINETUNED_MODEL_DIR} ...")
t0 = time.time()

ft_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    cfg.BASE_MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map="auto",
    token=cfg.HF_TOKEN,
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(ft_base, cfg.FINETUNED_MODEL_DIR)
ft_model.eval()

ft_processor = AutoProcessor.from_pretrained(
    cfg.FINETUNED_MODEL_DIR,
    token=cfg.HF_TOKEN,
    trust_remote_code=True,
)
print(f"✅ Fine-tuned model loaded  ({time.time()-t0:.1f}s)")

# ── 2. Base model ─────────────────────────────────────────────────────────
print(f"\n⏳ Loading base model  ({cfg.BASE_MODEL_NAME.split('/')[-1]}) ...")
t0 = time.time()
torch.cuda.empty_cache()

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    cfg.BASE_MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map="auto",
    token=cfg.HF_TOKEN,
    trust_remote_code=True,
)
base_model.eval()

base_processor = AutoProcessor.from_pretrained(
    cfg.BASE_MODEL_NAME,
    token=cfg.HF_TOKEN,
    trust_remote_code=True,
)
print(f"✅ Base model loaded  ({time.time()-t0:.1f}s)")

# ============================================================================
# INFERENCE HELPER
# ============================================================================

def run_inference(model, processor, image_path: str, prompt: str,
                  max_new_tokens: int = 256,
                  do_sample: bool = False) -> Tuple[str, float]:
    """
    Returns (prediction_text, inference_time_seconds).
    Handles missing images gracefully.
    """
    if not os.path.exists(image_path):
        return "[IMAGE NOT FOUND]", 0.0

    msgs = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text",  "text":  prompt},
    ]}]
    text     = processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    img_in, vid_in = process_vision_info(msgs)
    inputs   = processor(text=[text], images=img_in, videos=vid_in,
                         return_tensors="pt")

    dev    = next(model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )
    elapsed = time.time() - t0

    pred = processor.batch_decode(
        [out[0][inputs["input_ids"].shape[1]:]],
        skip_special_tokens=True,
    )[0].strip()

    return pred, elapsed

# ============================================================================
# LOAD TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("📂  LOADING TEST DATA")
print("=" * 80)

# Prefer test.json; fall back to val.json if missing
eval_json = cfg.TEST_JSON if os.path.exists(cfg.TEST_JSON) else cfg.VAL_JSON
label     = "test" if os.path.exists(cfg.TEST_JSON) else "val (fallback)"

with open(eval_json, "r") as f:
    raw_data = json.load(f)

# Filter to only samples whose images actually exist
test_data = [
    item for item in raw_data
    if os.path.exists(os.path.join(cfg.IMAGES_DIR, item["image"]))
]

print(f"   Source   : {label}  ({eval_json})")
print(f"   Total    : {len(raw_data)} samples in file")
print(f"   Usable   : {len(test_data)} samples (images found)")

if len(test_data) == 0:
    raise FileNotFoundError(
        f"No valid images found under {cfg.IMAGES_DIR}. "
        "Check IMAGES_DIR or TEST_JSON path.")

# ============================================================================
# EVALUATION LOOP
# ============================================================================

print("\n" + "=" * 80)
print("🔬  RUNNING EVALUATION")
print("=" * 80)
print(f"\n   Evaluating {len(test_data)} samples  (greedy, max {cfg.MAX_NEW_TOKENS} tokens)\n")

results       = []
t_eval_start  = time.time()

for idx, item in enumerate(test_data, 1):
    imgp   = os.path.join(cfg.IMAGES_DIR, item["image"])
    prompt = (item["conversations"][0]["value"]
              .replace("<image>\n", "").replace("<image>", "").strip())
    gt     = item["conversations"][1]["value"]

    # ── Base model inference ──────────────────────────────────────────────
    base_pred, base_time = run_inference(
        base_model, base_processor, imgp, prompt, cfg.MAX_NEW_TOKENS)
    base_m = compute_all_metrics(base_pred, gt)

    # ── Fine-tuned model inference ────────────────────────────────────────
    ft_pred, ft_time = run_inference(
        ft_model, ft_processor, imgp, prompt, cfg.MAX_NEW_TOKENS)
    ft_m = compute_all_metrics(ft_pred, gt)

    delta_sim = ft_m["similarity"] - base_m["similarity"]
    marker    = "🟢" if delta_sim > 0 else ("🔴" if delta_sim < -1 else "🟡")

    # ── Console output ────────────────────────────────────────────────────
    show = (cfg.VERBOSE_EVERY == 0) or (idx % cfg.VERBOSE_EVERY == 0) or (idx == 1)
    if show:
        print(f"─── [{idx:3d}/{len(test_data)}]  {item['image']} ───")
        print(f"   GT       : {gt[:100]}")
        print(f"   Base     : {base_pred[:100]}")
        print(f"   FT       : {ft_pred[:100]}")
        print(f"   Base sim : {base_m['similarity']:5.1f}%  "
              f"F1:{base_m['word_f1']:5.1f}%  "
              f"CER:{base_m['cer']:5.1f}%  "
              f"WER:{base_m['wer']:5.1f}%  "
              f"EM:{'✓' if base_m['exact_match'] else '✗'}")
        print(f"   FT sim   : {ft_m['similarity']:5.1f}%  "
              f"F1:{ft_m['word_f1']:5.1f}%  "
              f"CER:{ft_m['cer']:5.1f}%  "
              f"WER:{ft_m['wer']:5.1f}%  "
              f"EM:{'✓' if ft_m['exact_match'] else '✗'}")
        print(f"   {marker} Δ sim {delta_sim:+.1f}%  "
              f"| base {base_time:.1f}s  ft {ft_time:.1f}s\n")
    else:
        # one-liner for non-verbose steps
        print(f"[{idx:3d}/{len(test_data)}]  "
              f"base={base_m['similarity']:5.1f}%  "
              f"ft={ft_m['similarity']:5.1f}%  "
              f"{marker} Δ{delta_sim:+.1f}%  {item['image']}")

    results.append({
        "idx":              idx,
        "image":            item["image"],
        "ground_truth":     gt,
        "base_prediction":  base_pred,
        "ft_prediction":    ft_pred,
        # base metrics
        "base_similarity":  base_m["similarity"],
        "base_word_f1":     base_m["word_f1"],
        "base_cer":         base_m["cer"],
        "base_wer":         base_m["wer"],
        "base_exact_match": base_m["exact_match"],
        "base_inf_time":    round(base_time, 2),
        # fine-tuned metrics
        "ft_similarity":    ft_m["similarity"],
        "ft_word_f1":       ft_m["word_f1"],
        "ft_cer":           ft_m["cer"],
        "ft_wer":           ft_m["wer"],
        "ft_exact_match":   ft_m["exact_match"],
        "ft_inf_time":      round(ft_time, 2),
        # delta
        "delta_similarity": round(delta_sim, 2),
        "delta_word_f1":    round(ft_m["word_f1"] - base_m["word_f1"], 2),
        "delta_cer":        round(ft_m["cer"] - base_m["cer"], 2),
        "delta_wer":        round(ft_m["wer"] - base_m["wer"], 2),
    })

eval_time = time.time() - t_eval_start

# ============================================================================
# AGGREGATE METRICS
# ============================================================================

def avg(key): return float(np.mean([r[key] for r in results]))
def pct(key): return float(np.mean([r[key] for r in results])) * 100

base_avg = {
    "similarity":   avg("base_similarity"),
    "word_f1":      avg("base_word_f1"),
    "cer":          avg("base_cer"),
    "wer":          avg("base_wer"),
    "exact_match":  pct("base_exact_match"),
    "inf_time":     avg("base_inf_time"),
}
ft_avg = {
    "similarity":   avg("ft_similarity"),
    "word_f1":      avg("ft_word_f1"),
    "cer":          avg("ft_cer"),
    "wer":          avg("ft_wer"),
    "exact_match":  pct("ft_exact_match"),
    "inf_time":     avg("ft_inf_time"),
}

n_better = sum(1 for r in results if r["delta_similarity"] >  0.5)
n_worse  = sum(1 for r in results if r["delta_similarity"] < -0.5)
n_same   = len(results) - n_better - n_worse

# Score tiers for FT model
tier_counts = Counter(score_tier(r["ft_similarity"]) for r in results)

# Worst and best improvements
sorted_by_delta = sorted(results, key=lambda r: r["delta_similarity"])
top5_improved   = sorted_by_delta[-5:][::-1]
top5_degraded   = sorted_by_delta[:5]

print("\n" + "=" * 80)
print("📈  EVALUATION RESULTS")
print("=" * 80)

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│  METRIC              BASE MODEL      FINE-TUNED      DELTA      │
├─────────────────────────────────────────────────────────────────┤
│  Similarity %      {base_avg['similarity']:8.2f}       {ft_avg['similarity']:8.2f}      {ft_avg['similarity']-base_avg['similarity']:+7.2f}  │
│  Word F1 %         {base_avg['word_f1']:8.2f}       {ft_avg['word_f1']:8.2f}      {ft_avg['word_f1']-base_avg['word_f1']:+7.2f}  │
│  CER %  (↓ better) {base_avg['cer']:8.2f}       {ft_avg['cer']:8.2f}      {ft_avg['cer']-base_avg['cer']:+7.2f}  │
│  WER %  (↓ better) {base_avg['wer']:8.2f}       {ft_avg['wer']:8.2f}      {ft_avg['wer']-base_avg['wer']:+7.2f}  │
│  Exact Match %     {base_avg['exact_match']:8.2f}       {ft_avg['exact_match']:8.2f}      {ft_avg['exact_match']-base_avg['exact_match']:+7.2f}  │
│  Avg Infer (s)     {base_avg['inf_time']:8.2f}       {ft_avg['inf_time']:8.2f}      {ft_avg['inf_time']-base_avg['inf_time']:+7.2f}  │
└─────────────────────────────────────────────────────────────────┘
""")

print(f"   Samples evaluated  : {len(results)}")
print(f"   Better / Same / Worse : {n_better} / {n_same} / {n_worse}")
print(f"   Eval time          : {eval_time/60:.1f} min  "
      f"({eval_time/len(results):.1f}s per sample)")

print(f"\n📊 Fine-tuned score tiers:")
for tier, count in sorted(tier_counts.items(), reverse=True):
    bar = "█" * count
    print(f"   {tier:16s} : {count:3d}  {bar}")

print(f"\n🏆 Top 5 most improved samples:")
for r in top5_improved:
    print(f"   {r['image']:40s}  Δ {r['delta_similarity']:+.1f}%  "
          f"({r['base_similarity']:.1f}% → {r['ft_similarity']:.1f}%)")

print(f"\n⚠️  Top 5 most degraded samples:")
for r in top5_degraded:
    print(f"   {r['image']:40s}  Δ {r['delta_similarity']:+.1f}%  "
          f"({r['base_similarity']:.1f}% → {r['ft_similarity']:.1f}%)")

# ============================================================================
# VERDICT
# ============================================================================

sim_delta = ft_avg["similarity"] - base_avg["similarity"]
cer_delta = ft_avg["cer"]        - base_avg["cer"]
wer_delta = ft_avg["wer"]        - base_avg["wer"]

print(f"\n{'='*80}")
print("🎯  VERDICT")
print(f"{'='*80}")
if sim_delta > 5 and cer_delta < 0:
    print(f"🎉 STRONG IMPROVEMENT — fine-tuning significantly boosted OCR quality")
elif sim_delta > 0 and cer_delta <= 0:
    print(f"✅ POSITIVE GAIN — fine-tuning improved the model")
elif sim_delta > -2:
    print(f"🟡 NEUTRAL — fine-tuning had minimal effect; consider more data or epochs")
else:
    print(f"⚠️  REGRESSION — fine-tuned model underperforms base on this test set")

print(f"""
   Similarity  : {base_avg['similarity']:.2f}% → {ft_avg['similarity']:.2f}%  ({sim_delta:+.2f}%)
   Word F1     : {base_avg['word_f1']:.2f}% → {ft_avg['word_f1']:.2f}%  ({ft_avg['word_f1']-base_avg['word_f1']:+.2f}%)
   CER         : {base_avg['cer']:.2f}% → {ft_avg['cer']:.2f}%  ({cer_delta:+.2f}%)
   WER         : {base_avg['wer']:.2f}% → {ft_avg['wer']:.2f}%  ({wer_delta:+.2f}%)
   Exact Match : {base_avg['exact_match']:.1f}% → {ft_avg['exact_match']:.1f}%
""")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("=" * 80)
print("💾  SAVING RESULTS")
print("=" * 80)

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ── Full results CSV ──────────────────────────────────────────────────────
try:
    import pandas as pd
    cols = [
        "idx", "image",
        "base_similarity", "ft_similarity", "delta_similarity",
        "base_word_f1",    "ft_word_f1",    "delta_word_f1",
        "base_cer",        "ft_cer",        "delta_cer",
        "base_wer",        "ft_wer",        "delta_wer",
        "base_exact_match","ft_exact_match",
        "base_inf_time",   "ft_inf_time",
    ]
    df = pd.DataFrame(results)[cols]
    csv_path = f"{cfg.OUTPUT_DIR}/eval_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"   CSV    → {csv_path}")
except ImportError:
    print("   ⚠️  pandas not available — skipping CSV")

# ── Summary JSON ──────────────────────────────────────────────────────────
summary = {
    "model_finetuned":   cfg.FINETUNED_MODEL_DIR,
    "model_base":        cfg.BASE_MODEL_NAME,
    "eval_source":       label,
    "n_samples":         len(results),
    "eval_time_min":     round(eval_time / 60, 2),
    "base_model": {
        "avg_similarity":  round(base_avg["similarity"],  2),
        "avg_word_f1":     round(base_avg["word_f1"],     2),
        "avg_cer":         round(base_avg["cer"],         2),
        "avg_wer":         round(base_avg["wer"],         2),
        "exact_match_pct": round(base_avg["exact_match"], 2),
        "avg_inf_time_s":  round(base_avg["inf_time"],    2),
    },
    "finetuned_model": {
        "avg_similarity":  round(ft_avg["similarity"],    2),
        "avg_word_f1":     round(ft_avg["word_f1"],       2),
        "avg_cer":         round(ft_avg["cer"],           2),
        "avg_wer":         round(ft_avg["wer"],           2),
        "exact_match_pct": round(ft_avg["exact_match"],   2),
        "avg_inf_time_s":  round(ft_avg["inf_time"],      2),
    },
    "deltas": {
        "similarity":  round(sim_delta, 2),
        "word_f1":     round(ft_avg["word_f1"] - base_avg["word_f1"], 2),
        "cer":         round(cer_delta,  2),
        "wer":         round(wer_delta,  2),
        "exact_match": round(ft_avg["exact_match"] - base_avg["exact_match"], 2),
    },
    "sample_counts": {
        "better": n_better,
        "same":   n_same,
        "worse":  n_worse,
    },
    "score_tiers":  dict(tier_counts),
    "all_results":  results,
}

json_path = f"{cfg.OUTPUT_DIR}/eval_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"   JSON   → {json_path}")

# ── Clean-up ──────────────────────────────────────────────────────────────
del base_model, base_processor
torch.cuda.empty_cache()

print(f"""
================================================================================
✅  EVALUATION COMPLETE
================================================================================
  Samples    : {len(results)}
  Eval time  : {eval_time/60:.1f} min
  Similarity : {base_avg['similarity']:.2f}% → {ft_avg['similarity']:.2f}%  ({sim_delta:+.2f}%)
  Word F1    : {base_avg['word_f1']:.2f}% → {ft_avg['word_f1']:.2f}%
  CER        : {base_avg['cer']:.2f}% → {ft_avg['cer']:.2f}%  (lower = better)
  WER        : {base_avg['wer']:.2f}% → {ft_avg['wer']:.2f}%  (lower = better)
  Exact Match: {base_avg['exact_match']:.1f}% → {ft_avg['exact_match']:.1f}%
  Results    : {cfg.OUTPUT_DIR}/
================================================================================
""")

📊  EVALUATION  —  QWEN2.5-VL-3B  AUTONOMOUS  QLORA

🖥️  System:
   GPU 0: Tesla T4  (15.6 GB)
   GPU 1: Tesla T4  (15.6 GB)

📥  LOADING MODELS

⏳ Loading fine-tuned model from /kaggle/working/qwen3b_autonomous_qlora ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

✅ Fine-tuned model loaded  (7.6s)

⏳ Loading base model  (Qwen2.5-VL-3B-Instruct) ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Base model loaded  (7.0s)

📂  LOADING TEST DATA
   Source   : test  (/kaggle/working/finetune_dataset/test.json)
   Total    : 50 samples in file
   Usable   : 50 samples (images found)

🔬  RUNNING EVALUATION

   Evaluating 50 samples  (greedy, max 256 tokens)

─── [  1/50]  book_386_blurred.jpg ───
   GT       : Title: Virginibus Puerisque, and Other Papers by Robert Louis Stevenson
Author: Stevenson, Robert Lo
   Base     : 1. Book Title: Vegetables, Fruit, Peaches, and Other Fruits
2. Author Name: Robert Louis Stevenson
3
   FT       : Title: Virginian, Partrique, and Other Papers by Robert Louis Stevenson
Author: Stevenson, Robert Lo
   Base sim :  37.9%  F1: 38.1%  CER: 74.8%  WER:100.0%  EM:✗
   FT sim   :  95.7%  F1: 88.9%  CER:  5.0%  WER: 11.1%  EM:✗
   🟢 Δ sim +57.8%  | base 5.2s  ft 8.0s

[  2/50]  base= 21.4%  ft= 98.8%  🟢 Δ+77.4%  book_75_blurred.jpg
[  3/50]  base= 57.4%  ft=100.0%  🟢 Δ+42.6%  book_400_blurred.jpg
[  4/50]  base= 37.1%  ft=100.0%  🟢 Δ+62.9%  book_158_bl